# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedkhaled600/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

***Answer:***

**Finding 1 — "The Freshness Multiplier" (Finding #4), the paper's headline playbook action.**
The paper reports 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more
impressions, and ranks "refresh mature pages" as playbook action #1.

*My methodology question:* were refreshed pages chosen at random, or did an editor pick which
older pages to refresh? The paper doesn't say. Per the writing-honest-claims skill's own
selection-bias check: if the "treated" group was *chosen* (an editor picks pages worth the effort
-- likely pages that already had some remaining authority or demand), part of the 3.2x/57x gap is
the choosing, not the refresh itself. I'd ask: could a comparison between refreshed pages and a
*matched* set of similarly-aged, similarly-authoritative unrefreshed pages narrow the gap, and if
so, by how much? To the paper's credit, it already self-flags the adjacent 361+ freshness bucket
as unstable (283:1 ratio on just 1 declining page) -- I'm asking the same kind of question about a
different part of the same finding, in the same spirit the paper already models.

**Finding 2 — "What Predicts Growth?" (ML Appendix), logistic regression, 71% holdout accuracy.**
The label is growth/decline from 30d-vs-prior-30d impression trend; the methodology section says
"Random Forest (80/20 split), Logistic Regression (80/20 split)" with no mention of grouping by
brand.

*My methodology question:* was that 80/20 split random by row, or grouped by brand? This isn't a
hypothetical for me -- it's the exact question my own Week-5 model failed until I checked it (see
section 2 below). With 57 brands in the portfolio, a random split can let the same brand appear in
both train and test, which the section-2 result below shows inflates the score. I'd also ask where
exactly the growth/decline label's threshold sits relative to the *features* being used to predict
it: if any predictor overlaps with the same 30-day window used to define the label, the model
could be leaning on the label's own construction rather than a generalizable pattern -- the same
trap I found and fixed in my own model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

***Answer:***

Same final honest feature set from ML-08 (`prior_impressions_h1`, `prior_avg_position_h1`,
`content_age_days_h1_end`, `word_count`, `search_volume`, `backlinks`, `category_count`,
`has_keyword_data`, plus `content_type`/`main_intent`/`competition_level`), same `month=2026-03`
Random Forest -- trained once under a plain **random row split**, once under the **client-grouped
split**, to directly measure the exact risk I raised about the paper's finding 2 above.


***Answer:***

**Measured:**

| split | AUC | precision@10 | precision@20 | precision@50 | base_rate |
|---|---|---|---|---|---|
| BEFORE: random row split | 0.914 | 0.70 | 0.70 | 0.60 | 0.093 |
| AFTER: client-grouped split | 0.862 | 0.30 | 0.40 | 0.38 | 0.106 |

AUC gap: 0.052 -- modest on its own. But **precision@10 collapses from 0.70 to 0.30**, a 40-point
drop, while AUC barely moves. This is the concrete, quantified version of the methodology question
I raised in section 1 about the paper's growth-prediction model: a single "holdout accuracy" or
AUC number can look almost fine even when the same model falls apart exactly where it's used
operationally (the top of a ranked queue), if the split wasn't grouped by the right unit. Since I
don't know whether the paper's 80/20 split was grouped by brand, I can't assume its 71% accuracy
would survive the same kind of check -- which is precisely why I'm flagging it as a question, not
an accusation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb, pandas as pd, numpy as np
con = duckdb.connect()

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "fact_content_daily_performance/month=2026-03/*.parquet"
CUTOFF = "DATE '2026-03-15'"
SEED = 42

panel = con.sql(f"""
    WITH h1 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks)       AS prior_clicks_h1,
               SUM(gsc_impressions)  AS prior_impressions_h1,
               AVG(gsc_avg_position) AS prior_avg_position_h1
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date <= {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    ),
    h2 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS second_half_clicks
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date > {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT h1.*, h2.second_half_clicks,
           dc.content_type, dc.main_intent, dc.competition_level,
           dc.word_count, dc.search_volume, dc.backlinks, dc.category_count,
           {CUTOFF} - dc.content_created_date AS content_age_days_h1_end
    FROM h1
    JOIN h2 USING (client_hash_id, content_hash_id)
    JOIN read_parquet('{BASE}/dim_content.parquet') dc
        USING (client_hash_id, content_hash_id)
    WHERE dc.is_deleted = FALSE
""").df()

panel["declining"] = (panel["second_half_clicks"] < panel["prior_clicks_h1"]).astype(int)
panel["has_keyword_data"] = panel["search_volume"].notna().astype(int)
panel["search_volume"] = panel["search_volume"].fillna(0)
panel["backlinks"] = panel["backlinks"].fillna(0)

# ML-08's final honest feature set -- NOT prior_clicks_h1/prior_ctr_h1/days_since_update_h1_end
safe_num_cols = ["prior_impressions_h1", "prior_avg_position_h1", "content_age_days_h1_end",
                  "word_count", "search_volume", "backlinks", "category_count", "has_keyword_data"]
cat_cols = ["content_type", "main_intent", "competition_level"]

X_full = pd.get_dummies(panel[safe_num_cols + cat_cols], columns=cat_cols, drop_first=True).fillna(0)
y_full = panel["declining"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# --- BEFORE: plain random row split (same risk the paper's methodology doesn't rule out) ---
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X_full, y_full, test_size=0.2, random_state=SEED, stratify=y_full)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1).fit(Xtr_r, ytr_r)
score_random = rf_random.predict_proba(Xte_r)[:, 1]
auc_random = roc_auc_score(yte_r, score_random)

# --- AFTER: client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(panel, groups=panel["client_hash_id"]))
Xtr_g, Xte_g = X_full.iloc[train_idx], X_full.iloc[test_idx]
ytr_g, yte_g = y_full.iloc[train_idx], y_full.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1).fit(Xtr_g, ytr_g)
score_grouped = rf_grouped.predict_proba(Xte_g)[:, 1]
auc_grouped = roc_auc_score(yte_g, score_grouped)

rows = []
for name, scores, labels in [("BEFORE: random row split", score_random, yte_r),
                              ("AFTER: client-grouped split", score_grouped, yte_g)]:
    row = {"split": name, "AUC": round(roc_auc_score(labels, scores), 3)}
    for k in [10, 20, 50]:
        row[f"precision@{k}"] = round(precision_at_k(scores, labels, k), 3)
    row["base_rate"] = round(labels.mean(), 3)
    rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"AUC gap (random - grouped): {auc_random - auc_grouped:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                      split   AUC  precision@10  precision@20  precision@50  base_rate
   BEFORE: random row split 0.914           0.7           0.7          0.60      0.093
AFTER: client-grouped split 0.862           0.3           0.4          0.38      0.106

AUC gap (random - grouped): 0.052


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

***Answer:***

Same checklist from ML-05, applied to the FINAL feature set actually used above (8 numeric +
3 one-hot categoricals) -- confirming the fixes from ML-08 held, not just re-declaring them.


***Answer:***

**Measured: fully clean.** No label-source, ML-08-suspect, future-window, or product-flag columns
in the final feature matrix. Timeline holds (`2026-03-15 < 2026-03-16`, no overlap). Base rate
(0.093-0.106) sits nowhere near either AUC (0.862-0.914) -- neither number looks suspiciously
perfect, consistent with the honest (not leaky) result already established in ML-08.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. No label-source columns in the final feature matrix
label_source_cols = ["second_half_clicks", "declining"]
present = [c for c in label_source_cols if c in X_full.columns]
print("Label-source columns present in X_full (should be empty):", present)

# 2. No previously-diagnosed leakage suspects (ML-08's own trap)
suspect_cols = ["prior_clicks_h1", "prior_ctr_h1", "days_since_update_h1_end"]
present2 = [c for c in suspect_cols if c in X_full.columns]
print("ML-08 leakage suspects present in X_full (should be empty):", present2)

# 3. No future-window (h2 / second-half) columns
future_cols = [c for c in X_full.columns if "h2" in c or "second_half" in c]
print("Future-window columns present (should be empty):", future_cols)

# 4. No product-flag columns (confirmed absent from warehouse schema back in ML-04)
flag_cols = [c for c in X_full.columns if "health_score" in c or "flag" in c]
print("Product-flag columns present (should be empty):", flag_cols)

# 5. Timeline check -- every feature still comes from report_date <= cutoff
timeline = con.sql(f"""
    SELECT MAX(CASE WHEN report_date <= {CUTOFF} THEN report_date END) AS last_feature_date,
           MIN(CASE WHEN report_date >  {CUTOFF} THEN report_date END) AS first_label_date
    FROM read_parquet('{BASE}/{MONTH}')
""").df()
print()
print(timeline)
print("Timeline holds if last_feature_date < first_label_date.")

# 6. Base rate, printed next to the AUCs already reported in section 2
print()
print(f"Base rate: {y_full.mean():.3f} (compare against both AUCs above -- neither should be near 1.0)")


Label-source columns present in X_full (should be empty): []
ML-08 leakage suspects present in X_full (should be empty): []
Future-window columns present (should be empty): []
Product-flag columns present (should be empty): []

  last_feature_date first_label_date
0        2026-03-15       2026-03-16
Timeline holds if last_feature_date < first_label_date.

Base rate: 0.093 (compare against both AUCs above -- neither should be near 1.0)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

***Answer:***

**My boldest sentence (from ML-08's write-up):** *"The model that actually beats my baseline:
Random Forest (honest), at every K."*

**What's wrong with it:** "beats" and "at every K" both overreach. This is cross-sectional,
single-month, single-portfolio, one-time-split evidence -- not a validated claim that the model
will outperform the baseline going forward, on new months, or on clients outside this test split.
"At every K" is also an overclaim dressed as precision: I only checked K=10/20/50, not "every K."

**Rewritten in safe language (claim ladder):** *"On this held-out, client-grouped test split for
`month=2026-03`, the Random Forest model, with leakage-adjacent features removed, showed higher
precision than the ML-06 baseline rule at K=10, K=20, and K=50 (0.5 vs 0.2 at K=10, against a
10.6% base rate) -- a measured, decision-support result for this portfolio slice, not yet
validated on a different month or confirmed to generalize beyond this test split."*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Programmatic check: scan my own notebooks for banned words the writing-honest-claims skill flags
import re

banned = ["proves", "causes", "will increase", "the algorithm rewards", "predicted Google"]
my_claim_before = "The model that actually beats my baseline: Random Forest (honest), at every K."
my_claim_after = ("On this held-out, client-grouped test split for month=2026-03, the Random Forest "
                   "model, with leakage-adjacent features removed, showed higher precision than the "
                   "ML-06 baseline rule at K=10, K=20, and K=50 (0.5 vs 0.2 at K=10, against a 10.6% "
                   "base rate) -- a measured, decision-support result for this portfolio slice, not "
                   "yet validated on a different month or confirmed to generalize beyond this test split.")

for label, text in [("Before", my_claim_before), ("After", my_claim_after)]:
    hits = [w for w in banned if w.lower() in text.lower()]
    print(f"{label}: banned words found = {hits if hits else 'none'}")


Before: banned words found = none
After: banned words found = none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.